In [1]:
import numpy as np
import pandas as pd

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score


from multiprocessing import Pool, cpu_count

import dask
import dask.dataframe as dd
from dask.distributed import Client
client = Client(n_workers=20, memory_limit="10GB", interface='lo')

import dask_ml.cluster as dask_cluster

from pprint import pprint

/home/zwang937/anaconda3/lib/python3.7/site-packages/distributed/dashboard/core.py:72: UserWarning: 
Port 8787 is already in use. 
Perhaps you already have a cluster running?
Hosting the diagnostics dashboard on a random port instead.
  warnings.warn("\n" + msg)


### Import Overall Dataset

In [2]:
augmented_data_path = "../../data/augmented_us-counties-states_latest.csv"
#augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True)).compute()
augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True))
#augmented_df = pd.read_csv(augmented_data_path)

In [3]:
augmented_df.head(5)

,fips,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,...,totalTestsAntigen,totalTestsPeopleAntibody,totalTestsPeopleAntigen,totalTestsPeopleViral,totalTestsPeopleViralIncrease,totalTestsViral,totalTestsViralIncrease,metrics.testPositivityRatio,metrics.vaccinationsInitiatedRatio,metrics.vaccinationsCompletedRatio
0,1001.0,2021-10-02,Autauga,Alabama,570.0,142.0,2021-10-02,620.0,645.142857,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.125,0.430,0.338
1,1001.0,2023-03-01,Autauga,Alabama,229.0,232.0,2023-03-01,1135.0,205.857143,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.047,0.579,0.460
2,1001.0,2022-02-23,Autauga,Alabama,669.0,184.0,2022-02-23,764.0,1415.714286,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.094,0.548,0.433
3,1001.0,2020-04-20,Autauga,Alabama,22.0,1.0,2020-04-20,90.0,21.428571,32.532237,...,NaN,NaN,NaN,45900.0,188.0,NaN,0.0,0.051,0.000,0.000
4,1001.0,2022-05-12,Autauga,Alabama,103.0,216.0,2022-05-12,842.0,94.285714,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.057,0.558,0.444


In [4]:
### Import HSA HRR Dataset

In [5]:
hsa_hrr_data_path = "../../data/ZipHsaHrr15.csv"
hsa_hrr_df = pd.read_csv(hsa_hrr_data_path)
#(hsa_hrr_df).head(5)
ZIP_to_FIPS_path = "../../data/ZIP_to_FIPS.csv"
ZIP_to_FIPS_df = pd.read_csv(ZIP_to_FIPS_path)
#ZIP_to_FIPS_df.head(5)
hhs_regions_path = "../../data/hhs_regions.csv"
#hhs_regions_df = pd.read_csv(hhs_regions_path)
hhs_regions_df = dd.read_csv(hhs_regions_path, assume_missing=True)
hhs_regions_df = hhs_regions_df.drop(columns=["region", "regional_office"])
hhs_regions_df = hhs_regions_df.rename(columns={"region_number":"hhs_region"})
hhs_regions_df.head(5)

#HHS_FIPS_df = pd.merge(left=hsa_hrr_df, right=ZIP_to_FIPS_df, left_on="zipcode15", right_on="ZIP", how="inner")
#HHS_FIPS_df = HHS_FIPS_df.drop(columns=["zipcode15", "hsastate","STATE","COUNTYNAME"])
#HHS_FIPS_df.head(5)

,hhs_region,state_or_territory
0,1.0,Connecticut
1,1.0,Maine
2,1.0,Massachusetts
3,1.0,New Hampshire
4,1.0,Rhode Island


In [6]:
hhs_augmented_df=dd.merge(augmented_df,hhs_regions_df, left_on="state", right_on="state_or_territory", how="left")
hhs_augmented_df.head(5)

,fips,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,...,totalTestsPeopleAntigen,totalTestsPeopleViral,totalTestsPeopleViralIncrease,totalTestsViral,totalTestsViralIncrease,metrics.testPositivityRatio,metrics.vaccinationsInitiatedRatio,metrics.vaccinationsCompletedRatio,hhs_region,state_or_territory
0,1001.0,2021-10-02,Autauga,Alabama,570.0,142.0,2021-10-02,620.0,645.142857,32.532237,...,NaN,NaN,NaN,NaN,NaN,0.125,0.430,0.338,4.0,Alabama
1,1001.0,2023-03-01,Autauga,Alabama,229.0,232.0,2023-03-01,1135.0,205.857143,32.532237,...,NaN,NaN,NaN,NaN,NaN,0.047,0.579,0.460,4.0,Alabama
2,1001.0,2022-02-23,Autauga,Alabama,669.0,184.0,2022-02-23,764.0,1415.714286,32.532237,...,NaN,NaN,NaN,NaN,NaN,0.094,0.548,0.433,4.0,Alabama
3,1001.0,2020-04-20,Autauga,Alabama,22.0,1.0,2020-04-20,90.0,21.428571,32.532237,...,NaN,45900.0,188.0,NaN,0.0,0.051,0.000,0.000,4.0,Alabama
4,1001.0,2022-05-12,Autauga,Alabama,103.0,216.0,2022-05-12,842.0,94.285714,32.532237,...,NaN,NaN,NaN,NaN,NaN,0.057,0.558,0.444,4.0,Alabama


### Obtain non time-varying features to cluster upon

In [7]:
stds = hhs_augmented_df.groupby('fips').std().compute()
non_time_varying_filter = stds.select_dtypes(include=[np.number]).apply(lambda x: (x < 0.0001) | (x.isna())).all()
stds_filtered = stds.loc[:, non_time_varying_filter]

In [8]:
print("Out of {} columns in hhs_augmented_df, {} are non-time varying".format(len(hhs_augmented_df.columns),len(stds_filtered.columns)))
stds_filtered.columns
# Get rid of static time series data
non_time_varying_columns = [name for name in stds_filtered.columns if not any(date in name for date in ['2020', '2021'])]
print("Out of {} columns in hhs_augmented_df, {} are non-time varying".format(len(hhs_augmented_df.columns),len(non_time_varying_columns)))
stds_filtered[non_time_varying_columns]

Out of 422 columns in hhs_augmented_df, 213 are non-time varying
Out of 422 columns in hhs_augmented_df, 200 are non-time varying


,LAT,LON,E_UNEMP,E_PCI,E_SNGPNT,E_MUNIT,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,...,"Automatic_applications_sent_for_mail-in_ballots_in_response_to_COVID-19_(0,1,2_2_being_conditional-_see_notes_for_details)",Last_date_of_receipt_of_mail-in_ballot_request_for_the_general_election_(by_mail_or_online),Mention_of_tribal_casinos,Jan_1_2019_Minimum_Wage,Mar_29_2019_Minimum_Wage,Jul_1_2019_Minimum_Wage,Oct_1_2019_Minimum_Wage,Different_Minimum_Wage_for_Smaller_Businesses,positiveScore,hhs_region
fips,,,,,,,,,,,,,,,,,,,,,
1001.0,0.000000e+00,9.252000e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1003.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.151222e-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1005.0,4.647407e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.286213e-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1007.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1009.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69120.0,0.000000e+00,2.762932e-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78010.0,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
78020.0,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [9]:
take_col = [name for name in non_time_varying_columns if any(take in name for take in ['take'])]
take_col

[]

In [10]:
non_time_varying_columns

['LAT',
 'LON',
 'E_UNEMP',
 'E_PCI',
 'E_SNGPNT',
 'E_MUNIT',
 'E_CROWD',
 'E_NOVEH',
 'E_GROUPQ',
 'EP_POV',
 'MP_POV',
 'EP_UNEMP',
 'MP_UNEMP',
 'EP_PCI',
 'MP_PCI',
 'EP_NOHSDP',
 'MP_NOHSDP',
 'EP_AGE65',
 'MP_AGE65',
 'EP_AGE17',
 'MP_AGE17',
 'EP_DISABL',
 'MP_DISABL',
 'EP_SNGPNT',
 'MP_SNGPNT',
 'EP_MINRTY',
 'MP_MINRTY',
 'EP_LIMENG',
 'MP_LIMENG',
 'EP_MUNIT',
 'MP_MUNIT',
 'EP_MOBILE',
 'MP_MOBILE',
 'EP_CROWD',
 'MP_CROWD',
 'EP_NOVEH',
 'MP_NOVEH',
 'EP_GROUPQ',
 'MP_GROUPQ',
 'EPL_POV',
 'EPL_UNEMP',
 'EPL_PCI',
 'EPL_NOHSDP',
 'SPL_THEME1',
 'RPL_THEME1',
 'EPL_AGE65',
 'EPL_AGE17',
 'EPL_DISABL',
 'EPL_SNGPNT',
 'SPL_THEME2',
 'RPL_THEME2',
 'EPL_MINRTY',
 'EPL_LIMENG',
 'SPL_THEME3',
 'RPL_THEME3',
 'EPL_MUNIT',
 'EPL_MOBILE',
 'EPL_CROWD',
 'EPL_NOVEH',
 'EPL_GROUPQ',
 'SPL_THEME4',
 'RPL_THEME4',
 'SPL_THEMES',
 'RPL_THEMES',
 'F_POV',
 'F_UNEMP',
 'F_PCI',
 'F_NOHSDP',
 'F_THEME1',
 'F_AGE65',
 'F_AGE17',
 'F_DISABL',
 'F_SNGPNT',
 'F_THEME2',
 'F_MINRTY',
 'F_LIM

### K-means

In [11]:
# Define Hyperparameters
X = hhs_augmented_df[["fips", "county", "state"] + non_time_varying_columns]#.to_dask_array(lengths=True)
y = np.log(hhs_augmented_df["rolled_cases"] + 1)#.to_dask_array(lengths=True)

X_unique = X.drop_duplicates().compute()
X_unique = X_unique.dropna(subset=["positiveScore"]) 
X_unique.to_csv("hhs_kmeans_data.csv", index=False)

n_clusters_range = range(1, 11)
window_size_range = range(2,15)



In [12]:
print([stat for stat in X_unique.min()])
print([stat for stat in X_unique.max()])

[1001.0, 'Abbeville', 'Alabama', 19.597764, -164.188912, 0.0, 10148.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3, 0.2, 0.0, 0.1, 10148.0, 137.0, 1.2, 0.1, 3.8, 0.0, 5.3, 0.0, 3.8, 0.1, 0.0, 0.1, 0.0, 0.0, 0.0, 0.1, 0.0, 0.1, 0.0, 0.1, 0.0, 0.1, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0003, 0.0, 0.0669, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1372, 0.0, 0.0, 0.0, 0.0003, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1741, 0.0, 1.9371, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 1.7, 0.1, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 2.0, 2.0, 1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 235.0, 3.5, 125.0, 138.0, 130.0, 1.0, 0.0, 0.0, 7000.0, 0.0, 5.4, 180.86, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.11, 548.0